# 🚢 FreightQuote AI - Grand Schema Merge AutoML Pipeline
This notebook dynamically downloads all 3 mandatory Kaggle datasets, scans every CSV to find the target column, generates a synthetic dataset, and merges all 4 data sources into one massive master dataframe to train the 10-model competition.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

MODELS_DIR = '/content/drive/MyDrive/FreightQuote_AI/kaggle'
os.makedirs(MODELS_DIR, exist_ok=True)
print(f"Models will be saved to: {MODELS_DIR}")

!pip install -q scikit-learn pandas numpy joblib xgboost

Mounted at /content/drive
Models will be saved to: /content/drive/MyDrive/FreightQuote_AI/kaggle


## 1. Kaggle Authentication & Universal Grand Merger

In [ ]:
import os, glob, pandas as pd, numpy as np, joblib
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, r2_score
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier, ExtraTreesClassifier, RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.linear_model import LogisticRegression, Ridge, Lasso, ElasticNet
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.svm import SVC, SVR
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
import warnings
warnings.filterwarnings('ignore')

try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
    print("✅ Kaggle credentials successfully loaded from Colab Secrets.")
except Exception as e:
    print("⚠️ Kaggle secrets not found in Colab userdata.")

def generate_synthetic_fallback(target_keyword, task_type='classification', N=5000):
    print(f"🤖 Generating Synthetic Data for '{target_keyword}'...")
    np.random.seed(42)
    X = pd.DataFrame({
        'synth_feature_1': np.random.normal(100, 15, N),
        'synth_feature_2': np.random.uniform(0, 1, N),
        'synth_feature_3': np.random.randint(1, 10, N),
        'synth_feature_4': np.random.poisson(5, N)
    })
    noise = np.random.normal(0, 0.1, N)
    signal = (X['synth_feature_1']/100) + X['synth_feature_2'] + (X['synth_feature_3']/10) + noise

    if task_type == 'classification':
        y = (signal > np.median(signal)).astype(int)
    else:
        y = signal * 1000

    X['TARGET_VAR'] = y
    return X

def process_kaggle_datasets(dataset_list, target_keyword, task_type='classification'):
    all_dfs = []

    for dset in dataset_list:
        print(f"\n--- Attempting {dset} ---")
        os.system(f"kaggle datasets download -d {dset} --unzip -q")

        csv_files = glob.glob("*.csv")
        if not csv_files:
            print(f"❌ No CSV found for {dset}.")
            continue

        found_target = False
        for csvf in csv_files:
            try:
                df = pd.read_csv(csvf, encoding='utf-8', on_bad_lines='skip')
            except Exception as e:
                try:
                    df = pd.read_csv(csvf, encoding='latin1', on_bad_lines='skip')
                except:
                    continue

            target_col = None
            for col in df.columns:
                if target_keyword.lower() in str(col).lower():
                    target_col = col
                    break

            if target_col:
                print(f"✅ Target '{target_col}' identified in {csvf}")
                df.rename(columns={target_col: 'TARGET_VAR'}, inplace=True)
                for col in df.columns:
                    if df[col].dtype == 'object' and df[col].nunique() > 1000:
                        df.drop(col, axis=1, inplace=True)

                for col in df.columns:
                    if df[col].dtype == 'object':
                        df[col] = LabelEncoder().fit_transform(df[col].astype(str))

                if len(df) > 5000:
                    df = df.sample(n=5000, random_state=42)

                all_dfs.append(df)
                found_target = True
                break

        if not found_target:
            print(f"❌ Target keyword '{target_keyword}' not found in any CSVs inside {dset}.")

        os.system("rm -f *.csv")

    synth_df = generate_synthetic_fallback(target_keyword, task_type)
    all_dfs.append(synth_df)

    print(f"\n🔗 Merging {len(all_dfs)} data sources into master dataset...")
    master_df = pd.concat(all_dfs, ignore_index=True)
    master_df.fillna(0, inplace=True)

    print(f"Master Dataset Shape: {master_df.shape}")

    X = master_df.drop('TARGET_VAR', axis=1)
    y = master_df['TARGET_VAR']

    if task_type == 'classification':
        if len(y.unique()) > 10:
            y = (y > y.median()).astype(int)
        else:
            y = LabelEncoder().fit_transform(y)

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42)
    return Xtr, Xte, ytr, yte


✅ Kaggle credentials successfully loaded from Colab Secrets.


## Agent 1: Route Intelligence

In [ ]:
try:
    datasets = ["datasetengineer/logistics-and-supply-chain-dataset", "usdot/freight-analysis-framework", "nicolemachado/transportation-and-logistics-tracking-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Probability", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent1_route_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 1: Route Intelligence: {e}")



--- Attempting datasetengineer/logistics-and-supply-chain-dataset ---
✅ Target 'delay_probability' identified in dynamic_supply_chain_logistics_dataset.csv

--- Attempting usdot/freight-analysis-framework ---
❌ Target keyword 'Probability' not found in any CSVs inside usdot/freight-analysis-framework.

--- Attempting nicolemachado/transportation-and-logistics-tracking-dataset ---
❌ No CSV found for nicolemachado/transportation-and-logistics-tracking-dataset.
🤖 Generating Synthetic Data for 'Probability'...

🔗 Merging 2 data sources into master dataset...
Master Dataset Shape: (10000, 29)

✅ Saved best model (Gradient Boosting) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent1_route_model.joblib


## Agent 2: Dynamic Pricing

In [ ]:
try:
    datasets = ["apoorvwatsky/supply-chain-shipment-pricing-data", "shahriarkabir/us-logistics-performance-dataset", "ayeshaseherr/delivery-logistics-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Cost", "regression")

    if "regression" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "regression" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent2_freight_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 2: Dynamic Pricing: {e}")



--- Attempting apoorvwatsky/supply-chain-shipment-pricing-data ---
✅ Target 'Freight Cost (USD)' identified in SCMS_Delivery_History_Dataset_20150929.csv

--- Attempting shahriarkabir/us-logistics-performance-dataset ---
✅ Target 'Cost' identified in logistics_shipments_dataset.csv

--- Attempting ayeshaseherr/delivery-logistics-dataset ---
✅ Target 'delivery_cost' identified in Delivery_Logistics.csv
🤖 Generating Synthetic Data for 'Cost'...

🔗 Merging 4 data sources into master dataset...
Master Dataset Shape: (17000, 53)

✅ Saved best model (Neural Network) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent2_freight_model.joblib


## Agent 3: Weather Disruptions

In [ ]:
try:
    datasets = ["bwandowando/noaa-storm-events-database", "minhnhathuynh/seattle-flight-delay-and-weather-prediction-dataset", "preethgunasekaran/flight-delayweather-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Delay", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent3_weather_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 3: Weather Disruptions: {e}")



--- Attempting bwandowando/noaa-storm-events-database ---
❌ Target keyword 'Delay' not found in any CSVs inside bwandowando/noaa-storm-events-database.

--- Attempting minhnhathuynh/seattle-flight-delay-and-weather-prediction-dataset ---
✅ Target 'DEP_DELAY' identified in train.csv

--- Attempting preethgunasekaran/flight-delayweather-dataset ---
❌ Target keyword 'Delay' not found in any CSVs inside preethgunasekaran/flight-delayweather-dataset.
🤖 Generating Synthetic Data for 'Delay'...

🔗 Merging 2 data sources into master dataset...
Master Dataset Shape: (10000, 46)

✅ Saved best model (Gradient Boosting) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent3_weather_model.joblib


## Agent 4: Customs Compliance

In [ ]:
try:
    datasets = ["datasetengineer/logistics-and-supply-chain-dataset", "apoorvwatsky/supply-chain-shipment-pricing-data", "sid321axn/audit-data"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Clearance", "regression")

    if "regression" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "regression" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent4_customs_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 4: Customs Compliance: {e}")



--- Attempting datasetengineer/logistics-and-supply-chain-dataset ---
✅ Target 'customs_clearance_time' identified in dynamic_supply_chain_logistics_dataset.csv

--- Attempting apoorvwatsky/supply-chain-shipment-pricing-data ---
❌ Target keyword 'Clearance' not found in any CSVs inside apoorvwatsky/supply-chain-shipment-pricing-data.

--- Attempting sid321axn/audit-data ---
❌ Target keyword 'Clearance' not found in any CSVs inside sid321axn/audit-data.
🤖 Generating Synthetic Data for 'Clearance'...

🔗 Merging 2 data sources into master dataset...
Master Dataset Shape: (10000, 29)

✅ Saved best model (Neural Network) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent4_customs_model.joblib


## Agent 5: Margin Optimization

In [ ]:
try:
    datasets = ["yogape/logistics-operations-database", "programmer3/logistics-operations-and-risk-dataset", "shahriarkabir/us-logistics-performance-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Margin", "regression")

    if "regression" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "regression" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent5_margin_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 5: Margin Optimization: {e}")



--- Attempting yogape/logistics-operations-database ---
❌ Target keyword 'Margin' not found in any CSVs inside yogape/logistics-operations-database.

--- Attempting programmer3/logistics-operations-and-risk-dataset ---
❌ Target keyword 'Margin' not found in any CSVs inside programmer3/logistics-operations-and-risk-dataset.

--- Attempting shahriarkabir/us-logistics-performance-dataset ---
❌ Target keyword 'Margin' not found in any CSVs inside shahriarkabir/us-logistics-performance-dataset.
🤖 Generating Synthetic Data for 'Margin'...

🔗 Merging 1 data sources into master dataset...
Master Dataset Shape: (5000, 5)

✅ Saved best model (Ridge Regression) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent5_margin_model.joblib


## Agent 6: Client Support

In [ ]:
try:
    datasets = ["mansithummar67/171k-product-review-with-sentiment-dataset", "kundanbedmutha/customer-sentiment-dataset", "sujalsuthar/amazon-delivery-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Sentiment", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent6_support_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 6: Client Support: {e}")



--- Attempting mansithummar67/171k-product-review-with-sentiment-dataset ---
✅ Target 'Sentiment' identified in RATIO.csv

--- Attempting kundanbedmutha/customer-sentiment-dataset ---
✅ Target 'sentiment' identified in Customer_Sentiment.csv

--- Attempting sujalsuthar/amazon-delivery-dataset ---
❌ Target keyword 'Sentiment' not found in any CSVs inside sujalsuthar/amazon-delivery-dataset.
🤖 Generating Synthetic Data for 'Sentiment'...

🔗 Merging 3 data sources into master dataset...
Master Dataset Shape: (15000, 20)

✅ Saved best model (Gradient Boosting) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent6_support_model.joblib


## Agent 7: Cargo Insurance

In [ ]:
try:
    datasets = ["ifteshanajnin/carinsuranceclaimprediction-classification", "litvinenko630/insurance-claims", "easonlai/sample-insurance-claim-prediction-dataset"]
    Xtr, Xte, ytr, yte = process_kaggle_datasets(datasets, "Claim", "classification")

    if "classification" == "classification":
        models = {
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingClassifier(random_state=42),
            'Logistic Regression': LogisticRegression(max_iter=1000),
            'Neural Network': MLPClassifier(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeClassifier(max_depth=5),
            'SVM': SVC(probability=True),
            'KNN': KNeighborsClassifier(n_neighbors=5),
            'AdaBoost': AdaBoostClassifier(n_estimators=50, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=100, random_state=42),
            'Naive Bayes': GaussianNB()
        }
    else:
        models = {
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'Gradient Boosting': GradientBoostingRegressor(random_state=42),
            'Ridge Regression': Ridge(),
            'Lasso Regression': Lasso(),
            'SVR': SVR(kernel='rbf'),
            'Neural Network': MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500),
            'Decision Tree': DecisionTreeRegressor(max_depth=5),
            'ElasticNet': ElasticNet(),
            'KNN': KNeighborsRegressor(n_neighbors=5),
            'Extra Trees': ExtraTreesRegressor(n_estimators=100, random_state=42)
        }

    best_score = -99
    best_pipe = None
    best_name = ""
    for name, m in models.items():
        try:
            pipe = Pipeline([('s', StandardScaler()), ('m', m)])
            pipe.fit(Xtr, ytr)
            score = accuracy_score(yte, pipe.predict(Xte)) if "classification" == "classification" else r2_score(yte, pipe.predict(Xte))
            if score > best_score:
                best_score = score
                best_pipe = pipe
                best_name = name
        except Exception as model_e:
            pass

    model_path = f"{MODELS_DIR}/agent7_insurance_model.joblib"
    joblib.dump(best_pipe, model_path)
    print(f"\n✅ Saved best model ({best_name}) to {model_path}")
except Exception as e:
    print(f"❌ Failed to process Agent 7: Cargo Insurance: {e}")



--- Attempting ifteshanajnin/carinsuranceclaimprediction-classification ---
✅ Target 'is_claim' identified in train.csv

--- Attempting litvinenko630/insurance-claims ---
✅ Target 'claim_status' identified in Insurance claims data.csv

--- Attempting easonlai/sample-insurance-claim-prediction-dataset ---
✅ Target 'insuranceclaim' identified in insurance3r2.csv
🤖 Generating Synthetic Data for 'Claim'...

🔗 Merging 4 data sources into master dataset...
Master Dataset Shape: (16338, 60)

✅ Saved best model (Gradient Boosting) to /content/drive/MyDrive/FreightQuote_AI/kaggle/agent7_insurance_model.joblib
